# 00 · Method, conventions and transform validation

**Scientific question.** Are the discrete transforms used throughout this project the ones the physics requires, and are their conventions pinned down unambiguously?

**Scope.** Grid definitions, state encoding, the QFT/FFT convention, the DST-II convention, and validation of both against closed-form expressions rather than against another library call.

**Inputs.** `configs/{PROFILE}.yaml` (loaded below). No other notebook needs to have
been run first: this notebook imports everything it needs from
`src/boundary_aware_dynamics` and holds no state from any other notebook.

**Expected outputs.** Printed validation errors for each convention. No files are written.

**Approximate runtime.** under 10 seconds on the `smoke` profile.

**Method.** Each transform is compared against an analytical matrix built from its defining formula. The QST circuit is compared against the analytical DST-II matrix on the data subspace.

**Assumptions.** Power-of-two grids; hbar and mass in simulation units taken from the configuration.

**References.** See `references/references.bib` and `docs/SCIENTIFIC_METHOD.md`.

**What this notebook does _not_ establish.** Nothing about accuracy of the dynamics. This notebook validates representations only.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from boundary_aware_dynamics.config import load_config
from boundary_aware_dynamics import plotting

PROFILE = os.environ.get("BAD_PROFILE", "smoke")   # "paper" for manuscript numbers
                                                   # (scripts/execute_notebooks.py sets this)
config = load_config(ROOT / "configs" / f"{PROFILE}.yaml")
plotting.apply_style("preview")
print(f"profile={config.profile}  config_hash={config.config_hash}")

## Grids and the two boundary models

The periodic grid excludes its right endpoint; the Dirichlet grid samples cell midpoints and excludes both walls. The midpoint offset is what makes the orthonormal DST-II diagonalise the Dirichlet Laplacian exactly.

In [ ]:
from boundary_aware_dynamics.grids import (
    periodic_grid, dirichlet_midpoint_grid, signed_frequency_indices, sine_mode_indices)

ring = periodic_grid(-8.0, 8.0, 8)
box = dirichlet_midpoint_grid(10.0, 8)
print("periodic :", np.round(ring.positions, 3))
print("dirichlet:", np.round(box.positions, 3))
print("first/last midpoint vs walls:", box.positions[0], box.positions[-1], "in (0,", box.length, ")")
print("signed FFT indices:", signed_frequency_indices(8).astype(int))
print("sine mode indices :", sine_mode_indices(8).astype(int), "<- starts at 1, not 0")

## Physical samples versus quantum amplitudes

$|\Psi\rangle=\sqrt{\Delta x}\sum_j \psi(x_j)|j\rangle$. A quadrature-normalised wavefunction is **not** a legal statevector; the conversion is explicit.

In [ ]:
from boundary_aware_dynamics.states import (
    gaussian_wavepacket, physical_to_amplitudes, quadrature_norm, euclidean_norm)

psi = gaussian_wavepacket(ring.positions, ring.spacing, centre=0.0, momentum=1.0, sigma=1.0)
amp = physical_to_amplitudes(psi, ring.spacing)
print(f"quadrature norm of physical samples : {quadrature_norm(psi, ring.spacing):.12f}")
print(f"euclidean  norm of physical samples : {euclidean_norm(psi):.12f}  <- not a statevector")
print(f"euclidean  norm of amplitudes       : {euclidean_norm(amp):.12f}")

## QFT convention

Qiskit's `QFTGate` carries the opposite exponential sign to NumPy's forward FFT, so the forward transform is `QFTGate(n).inverse()`. It includes the terminating swaps, so no manual bit reversal is needed.

In [ ]:
from qiskit.quantum_info import Operator
from qiskit.circuit.library import QFTGate
from boundary_aware_dynamics.transforms import dft_matrix

for n in (2, 3, 4):
    err = np.abs(Operator(QFTGate(n).inverse()).data - dft_matrix(2**n)).max()
    print(f"n={n}: || QFT^-1 - forward DFT ||_max = {err:.2e}")

## DST-II convention, validated against the closed form

$S[\nu-1,j]=\sqrt{2/N}\,\sin\!\big(\pi\nu(j+\tfrac12)/N\big)$, with an extra $1/\sqrt2$ on the Nyquist row $\nu=N$. Validating SciPy against this closed form is what pins the convention; validating it against another SciPy call would not.

In [ ]:
from boundary_aware_dynamics.transforms import (
    dst2_matrix, analytical_dst2_matrix, dst2_forward, analytical_sine_mode)

for n in (8, 16, 32):
    print(f"N={n:3d}: ||scipy DST-II - closed form|| = "
          f"{np.abs(dst2_matrix(n) - analytical_dst2_matrix(n)).max():.2e}")

S = dst2_matrix(16)
print("orthogonality  ||S Sᵀ - I|| =", f"{np.abs(S @ S.T - np.eye(16)).max():.2e}")

# Mode nu must land in output bin nu-1.
g = dirichlet_midpoint_grid(10.0, 16)
for nu in (1, 5, 16):
    samples = analytical_sine_mode(nu, 16, 10.0) * np.sqrt(g.spacing)
    print(f"  mode nu={nu:2d} -> peak bin {int(np.argmax(np.abs(dst2_forward(samples))))} (expected {nu-1})")

## The QST circuit is a real DST-II, not a relabelled QFT

The circuit odd-extends the register onto a $4N$-point grid, applies the QFT there, and applies the kinetic phase through the triangle-wave mode map. Its action on the data subspace is compared against $S^{\mathsf{T}}e^{-iE\Delta t/\hbar}S$ built from the closed-form matrix.

In [ ]:
from boundary_aware_dynamics.circuits.qst import (
    qst_kinetic_propagator_circuit, reference_dirichlet_kinetic_operator, extended_register_size)

L, mass, hbar, dt = 10.0, config.physics.mass, config.physics.hbar, 0.1
for n in (2, 3):
    N = 2**n
    n_data, n_total = extended_register_size(N)
    idx = 2 * np.arange(N)                       # |j> with both ancillas in |0>
    circuit = qst_kinetic_propagator_circuit(N, L, mass, hbar, dt)
    observed = Operator(circuit).data[np.ix_(idx, idx)]
    expected = reference_dirichlet_kinetic_operator(N, L, mass, hbar, dt)
    print(f"N={N}: data {n_data} + ancilla 2 = {n_total} qubits;  "
          f"|| circuit - analytical DST-II propagator || = {np.linalg.norm(observed - expected):.2e}")

In [ ]:
# A bare QFT on the same register is NOT a sine transform. The earlier version of
# this repository labelled one as "QST"; the disagreement is as large as the target.
from qiskit import QuantumCircuit

for n in (2, 3):
    N = 2**n
    qc = QuantumCircuit(n + 2)
    qc.append(QFTGate(n + 2).inverse(), range(n + 2))
    proxy = Operator(qc).data[:N, :N]
    S = analytical_dst2_matrix(N)
    print(f"N={N}: ||bare QFT - DST-II|| = {np.linalg.norm(proxy - S):.3f}  "
          f"(||DST-II|| = {np.linalg.norm(S):.3f})")

## Structured phase synthesis

Every diagonal here is quadratic in the register index, and register bits satisfy $b^2=b$, so each factorises into a global phase, $n$ single-qubit phases and $n(n-1)/2$ controlled phases — $O(n^2)$ rather than $O(2^n)$.

In [ ]:
from boundary_aware_dynamics.circuits.phases import (
    harmonic_position_expansion, signed_momentum_expansion,
    folded_sine_kinetic_expansion, linear_tilt_expansion, evaluate_bit_expansion)
from boundary_aware_dynamics.circuits.qft import periodic_kinetic_phase_diagonal
from boundary_aware_dynamics.circuits.qst import sine_mode_phase_diagonal
from boundary_aware_dynamics.propagators import harmonic_potential, tilted_potential

N, tau = 16, 0.137
gr = periodic_grid(-8.0, 8.0, N); gd = dirichlet_midpoint_grid(L, N)
checks = {
  "harmonic x^2": (harmonic_position_expansion(N, -8.0, gr.spacing, mass, 1.0, hbar, tau),
                   np.exp(-1j * harmonic_potential(gr.positions, mass, 1.0) * tau / hbar)),
  "signed p^2  ": (signed_momentum_expansion(N, gr.spacing, mass, hbar, tau),
                   periodic_kinetic_phase_diagonal(N, gr.spacing, mass, hbar, tau)),
  "folded sine ": (folded_sine_kinetic_expansion(N, L, mass, hbar, tau),
                   sine_mode_phase_diagonal(N, L, mass, hbar, tau)),
  "linear tilt ": (linear_tilt_expansion(N, L, 5.0, hbar, tau),
                   np.exp(-1j * tilted_potential(gd.positions, L, 5.0) * tau / hbar)),
}
for label, (expansion, exact) in checks.items():
    err = np.abs(evaluate_bit_expansion(expansion) - exact).max()
    print(f"{label}: error {err:.2e}   gates: {expansion.n_single_qubit_terms} 1q + "
          f"{expansion.n_two_qubit_terms} 2q   (generic diagonal would need ~{2**expansion.n_qubits})")

## Summary

**Main findings.** All four transform conventions match their closed forms to machine precision. The QST circuit reproduces the analytical DST-II propagator to below 1e-10 using two ancillas that are uncomputed unitarily. A bare QFT on the same register disagrees with the DST-II by an amount comparable to the transform itself, so it is not a usable proxy.

**Validation checks performed.** QFT sign and endianness against the explicit DFT matrix; DST-II against its closed form; orthogonality; mode-to-bin assignment; QST circuit against the analytical propagator; four structured phase expansions against their exact diagonals.

**Limitations.** Circuit validation is done at 2–3 data qubits, where the full unitary can be formed. Larger registers are covered by the resource analysis, not by unitary comparison.

**Generated files.** None. This notebook is a validation gate.

**Relationship to the manuscript.** Supports the methods section: every convention quoted there is fixed here, and the QST implementation status is established.

**Next.** `01_harmonic_oscillator.ipynb` uses these conventions for the first dynamical benchmark.